# GL Code Apply

Run this on new UPS files. It fills column M with a GL code. If the rules are not confident, it writes `review` instead.

How to use:
1. Put the new UPS csv files in `input/`. The file name needs CSI / Cooper / Belimo in it.
2. Run all.
3. Check `output/`:
   - `<name>_GL.csv` is the original file with column M filled in.
   - `<name>_GL_review.xlsx` lists the review rows, with the suggested GL and the reason.

The rules come from `GL_Code_Automation.ipynb` (v2).

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)


def col_idx(letter):
    # excel column letter -> 0-based index ('M' -> 12, 'AQ' -> 42)
    n = 0
    for ch in letter.strip().upper():
        n = n * 26 + (ord(ch) - 64)
    return n - 1


def read_ups_csv(path):
    # UPS files have no header row, read everything as text so nothing gets reformatted
    for enc in ["utf-8", "latin1"]:
        try:
            df = pd.read_csv(
                path, header=None, dtype=str, keep_default_na=False, encoding=enc
            )
            df.attrs["encoding"] = enc
            return df
        except UnicodeDecodeError:
            pass
    raise ValueError("could not read " + str(path))


def normalize_gl(code):
    # long dash -> '--', and 'Split 654...' / 'Split-654...' -> 'Split-654...'
    if not isinstance(code, str):
        return code
    code = code.strip()
    code = re.sub(r"[\u2014\u2015]", "--", code)
    code = re.sub(r"[\u2010\u2011\u2012\u2013]", "-", code)
    code = re.sub(r"^split[\s\-_:]*", "Split-", code, flags=re.I)
    return code


def get_col(df, c):
    return df[col_idx(c)].astype(str).str.strip()


def make_pattern(vals, word=False):
    if isinstance(vals, str):
        vals = [vals]
    alt = "|".join(re.escape(str(v)) for v in vals)
    if word:
        return r"(?<![A-Za-z0-9])(?:" + alt + r")(?![A-Za-z0-9])"
    return "(?:" + alt + ")"

In [ ]:
INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output")
HISTORY_DIR = Path("history")
for d in [INPUT_DIR, OUTPUT_DIR, HISTORY_DIR]:
    d.mkdir(exist_ok=True)

REVIEW_LABEL = "review"
KEEP_EXISTING_M = True  # don't overwrite column M if it already has a value


def detect_company(path):
    name = Path(path).name.upper()
    if "CSI" in name or "COOPER" in name:
        return "CSI"
    if "BELIMO" in name:
        return "Belimo"
    raise ValueError("can't tell the company from file name: " + Path(path).name)


def is_answer_key(path):
    name = Path(path).name
    return "_Completed_" in name and "Not_Completed" not in name

In [ ]:
# a rule is a list of conditions (col, op, value), all of them have to be true
# "P|Q" means column P or column Q
OPS = [
    "eq",
    "in",
    "contains",
    "word",
    "not_contains",
    "contains_any",
    "not_contains_any",
    "eq_col",
    "ne_col",
]


def cond_mask(df, cols, op, val):
    if op in ["eq_col", "ne_col"]:
        # compare two columns, e.g. sender city BS vs receiver city CA
        a = get_col(df, cols).str.lower()
        b = get_col(df, val).str.lower()
        return np.asarray(a == b) if op == "eq_col" else np.asarray(a != b)
    masks = []
    for c in cols.split("|"):
        s = get_col(df, c)
        if op == "eq":
            m = s.str.lower() == str(val).strip().lower()
        elif op == "in":
            m = s.str.lower().isin([str(v).strip().lower() for v in val])
        elif op == "word":
            m = s.str.contains(make_pattern(val, word=True), case=False, regex=True)
        else:  # contains / not_contains / contains_any / not_contains_any
            m = s.str.contains(make_pattern(val), case=False, regex=True)
        masks.append(np.asarray(m))
    if op.startswith("not_"):
        return ~np.logical_or.reduce(masks)
    return np.logical_or.reduce(masks)


def rule_mask(df, rule):
    m = np.ones(len(df), dtype=bool)
    for cols, op, val in rule["when"]:
        m &= cond_mask(df, cols, op, val)
    return m


def match_rules(df, rules):
    # for each row pick the matching rule with the highest priority
    # (ties -> the one listed first). conflict = top-priority rules disagree on the GL
    rules = [r for r in rules if r.get("enabled", True)]
    prio = np.array([r["priority"] for r in rules])
    gls = np.array([r["gl"] for r in rules], dtype=object)
    ids = np.array([r["id"] for r in rules], dtype=object)
    hits = np.column_stack([rule_mask(df, r) for r in rules])
    p = np.where(hits, prio, -1)
    best = p.max(axis=1)
    top = hits & (p == best[:, None])
    win = top.argmax(axis=1)
    matched = hits.any(axis=1)
    return pd.DataFrame(
        {
            "gl": np.where(matched, gls[win], ""),
            "rule": np.where(matched, ids[win], ""),
            "conflict": np.array([len(set(gls[row])) > 1 for row in top]),
            "all_rules": [";".join(ids[row]) for row in hits],
        }
    )

## Rules (edit here)

In [ ]:
OWN = {
    "CSI": ["cooper"],
    "Belimo": ["belimo", "warren"],
}  # words that mean "our company" in BP/BX

U527 = "65420-3000--CSI_U100-L527--"
U528 = "65420-3000--CSI_U100-L528--"
U529R = "65420-3000--CSI_U100-L529--"
COMPL = "65415-1800--CSI_U100-L529--"
IT = "65415-1804--CSI_U100-L529--"
SVC = "65415-5500--CSI_U100-L529--"
PUR = "65440-8401--CSI_M100-L510--"
ICO = "51630-1000--CSI_M100-L510--"
FROM_SUPPLIER = [
    ("BX", "contains_any", OWN["CSI"]),
    ("BP", "not_contains_any", OWN["CSI"]),
    ("AJ", "in", ["FC", "TP"]),
]

# layer: 0 default, 1-2 direction + site, 3 charge type, 4+ purpose (dept, person, reference)
# higher layer wins
RULES_V2 = {}
RULES_V2["CSI"] = [
    {
        "id": "C-DEF",
        "layer": 0,
        "when": [],
        "gl": U527,
        "why": "default: DFW outbound to customers",
    },
    {
        "id": "C-OUT-TRB",
        "layer": 1,
        "when": [("BP", "contains_any", OWN["CSI"]), ("BS", "contains", "Trumb")],
        "gl": U528,
        "why": "shipped from Trumbull",
    },
    {
        "id": "C-OUT-TON",
        "layer": 1,
        "when": [("BP", "contains_any", OWN["CSI"]), ("BS", "contains", "Tonawanda")],
        "gl": COMPL,
        "why": "shipped from North Tonawanda (complaints/recalls)",
    },
    {
        "id": "C-IN-RET",
        "layer": 1,
        "when": [
            ("BX", "contains_any", OWN["CSI"]),
            ("BP", "not_contains_any", OWN["CSI"]),
        ],
        "gl": COMPL,
        "why": "sent to Cooper by outside party, assume customer return",
    },
    {
        "id": "C-IN-PUR-TRB",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains_any", ["Trumb", "Worth"])],
        "gl": PUR,
        "why": "supplier to Trumbull/DFW",
    },
    {
        "id": "C-IN-PUR-SGL",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains", "Sugar Land")],
        "gl": "65440-8401--CSI_M100-L560--",
        "why": "supplier to Sugar Land",
    },
    {
        "id": "C-IN-PUR-STF",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains", "Stafford")],
        "gl": "65440-8401--CSI_M100-L562--",
        "why": "supplier to Stafford",
    },
    {
        "id": "C-IN-PUR-STF-PO",
        "layer": 3,
        "when": FROM_SUPPLIER
        + [("CA", "contains", "Stafford"), ("P", "contains", "550-0")],
        "gl": PUR,
        "why": "Stafford, PO550 number in P (Trumbull PO)",
    },
    {
        "id": "C-IN-PUR-STF-3P",
        "layer": 2,
        "when": [("BX", "contains", "LONESTAR"), ("CA", "contains", "Stafford")],
        "gl": "65440-8401--CSI_M100-L562--",
        "why": "supplier to Stafford (LoneStar)",
    },
    {
        "id": "C-IN-PUR-TON",
        "layer": 2,
        "when": FROM_SUPPLIER + [("CA", "contains", "Tonawanda")],
        "gl": "65440-8401--CSI_P100-L551--",
        "why": "supplier to North Tonawanda",
    },
    {
        "id": "C-IMP",
        "layer": 2,
        "when": [("BX", "contains_any", OWN["CSI"]), ("AJ", "eq", "IMP")],
        "gl": "Split-" + PUR,
        "why": "import -> split",
    },
    {
        "id": "C-ICO-INTL",
        "layer": 2,
        "when": [("BX", "contains_any", OWN["CSI"]), ("CD", "in", ["NL", "CR"])],
        "gl": ICO,
        "why": "to Cooper entity abroad (Herten / Alajuela)",
    },
    {
        "id": "C-SVC",
        "layer": 3,
        "when": [("AQ", "in", ["SVC", "AUD", "PRT"])],
        "gl": SVC,
        "why": "UPS service / audit / print charge",
    },
    {
        "id": "C-TAX",
        "layer": 3,
        "when": [("AQ", "eq", "TAX")],
        "gl": U527,
        "why": "tax",
    },
    {
        "id": "C-SO-TRB",
        "layer": 4,
        "when": [("P", "contains", "SO55000"), ("BS", "contains", "Trumb")],
        "gl": U528,
        "why": "Trumbull sales order",
    },
    {
        "id": "C-SO-DFW",
        "layer": 4,
        "when": [("P", "contains", "SO55000"), ("BS", "contains", "Worth")],
        "gl": U527,
        "why": "DFW sales order (incl. overseas distributor)",
    },
    {
        "id": "C-IT-ACCT",
        "layer": 4,
        "when": [("C", "eq", "0000XH8253")],
        "gl": IT,
        "why": "IT UPS account",
    },
    {
        "id": "C-IT-BW",
        "layer": 4,
        "when": [("BW", "contains_any", ["IT Dep", "help desk", "repair"])],
        "gl": IT,
        "why": "IT dept / repair",
    },
    {
        "id": "C-IT-AM",
        "layer": 4,
        "when": [("AM", "in", ["helpdesk911", "BBryanCSI1", "rquin19", "EdMak"])],
        "gl": IT,
        "why": "IT user id",
    },
    {
        "id": "C-P-1804",
        "layer": 4,
        "when": [("P", "eq", "83350-46")],
        "gl": IT,
        "why": "reference 83350-46",
    },
    {
        "id": "C-SVC-AM",
        "layer": 4,
        "when": [("AM", "in", ["mcard636", "lc963", "A.MollicaCSI"])],
        "gl": SVC,
        "why": "user id (answer key says 5500, doc said 1804)",
    },
    {
        "id": "C-P-5500",
        "layer": 4,
        "when": [("P", "eq", "83350-80")],
        "gl": SVC,
        "why": "reference 83350-80",
    },
    {
        "id": "C-RMA",
        "layer": 4,
        "when": [("BW", "word", ["RMA", "Returns", "Complaint"])],
        "gl": U529R,
        "why": "RMA / returns dept",
    },
    {
        "id": "C-RMA-RECALL",
        "layer": 5,
        "when": [("BW", "contains", "RECA")],
        "gl": COMPL,
        "why": "RMA ... RECALL = recall",
    },
    {
        "id": "C-RMA-AM",
        "layer": 4,
        "when": [("AM", "in", ["MThomasCSI1"])],
        "gl": U529R,
        "why": "returns user id",
    },
    {
        "id": "C-L528-AM",
        "layer": 4,
        "when": [("AM", "in", ["Sheana Elliott", "TMutambayi1"])],
        "gl": U528,
        "why": "user id",
    },
    {
        "id": "C-COMPL",
        "layer": 4,
        "when": [("P", "contains_any", ["Complaint", "Recall", "83350-45"])],
        "gl": COMPL,
        "why": "complaint / recall",
    },
    {
        "id": "C-QA",
        "layer": 5,
        "when": [("P", "contains", "QA Dept")],
        "gl": "65415-8440--CSI_M100-L510--",
        "why": "QA dept",
    },
    {
        "id": "C-P-6000",
        "layer": 4,
        "when": [("P", "eq", "83350-60")],
        "gl": "65415-6000--CSI_U100-L529--",
        "why": "reference 83350-60",
    },
    {
        "id": "C-P-15",
        "layer": 4,
        "when": [("P", "eq", "83350-15")],
        "gl": "65410-3000--CSI_U100-L529--",
        "why": "answer key differs from doc (doc: 65415-8401--CSI_M100-L510--)",
    },
    {
        "id": "C-SHOW",
        "layer": 4,
        "when": [
            (
                "AM|BW|BX",
                "contains_any",
                ["vdh510", "Tara Luczak", "Maria Tobin", "Drake Hotel", "AUGS"],
            )
        ],
        "gl": "65415-2000--CSI_M100-L529--",
        "why": "trade show / conference",
    },
    {
        "id": "C-LYNN",
        "layer": 4,
        "when": [("BP|BX", "contains", "Lynn Sales")],
        "gl": "65415-2113--CSI_U100-L529--",
        "why": "AL-Lynn Sales rep",
    },
]

RULES_V2["Belimo"] = [
    {
        "id": "B-DEF",
        "layer": 0,
        "when": [],
        "gl": "353500",
        "why": "default: outbound to customers",
    },
    {
        "id": "B-OUT-CA",
        "layer": 1,
        "when": [("CD", "eq", "CA"), ("BX", "not_contains_any", OWN["Belimo"])],
        "gl": "353550",
        "why": "export to Canada (not an affiliate)",
    },
    {
        "id": "B-IN",
        "layer": 1,
        "when": [
            ("BX", "contains_any", OWN["Belimo"]),
            ("BP", "not_contains_any", OWN["Belimo"]),
        ],
        "gl": "356200",
        "why": "supplier to Belimo",
    },
    {
        "id": "B-ICO",
        "layer": 2,
        "when": [
            ("BX", "contains", "belimo"),
            ("BP", "contains_any", OWN["Belimo"]),
            ("BS", "ne_col", "CA"),
        ],
        "gl": "356205",
        "why": "between Belimo sites / affiliates, or Warren to Belimo",
    },
    {
        "id": "B-MKT",
        "layer": 3,
        "when": [("BP", "contains", "imprint")],
        "gl": "369600",
        "why": "marketing: 4imprint",
    },
    {
        "id": "B-MKT2",
        "layer": 3,
        "when": [
            ("BW|P", "contains_any", ["Marissa", "BUSINES CARD", "Field Support"])
        ],
        "gl": "369600",
        "why": "marketing people / business cards",
    },
    {
        "id": "B-RMA",
        "layer": 4,
        "when": [("P", "word", "RMA")],
        "gl": "353590",
        "why": "RMA",
    },
    {
        "id": "B-RMA2",
        "layer": 4,
        "when": [("P", "contains", "2-10")],
        "gl": "353590",
        "why": "2-10xxxx is an RMA number (doc said 367230)",
    },
    {
        "id": "B-SAMPLE",
        "layer": 4,
        "when": [("P", "contains", "sample")],
        "gl": "367211",
        "why": "samples",
    },
    {
        "id": "B-TRADE",
        "layer": 4,
        "when": [("P", "contains", "trade")],
        "gl": "367400",
        "why": "trade show",
    },
    {
        "id": "B-PROMO",
        "layer": 5,
        "when": [("P", "contains", "trade"), ("P", "contains", "promo")],
        "gl": "367210",
        "why": "trade show promo",
    },
    {
        "id": "B-LIT",
        "layer": 6,
        "when": [("P", "word", "lit")],
        "gl": "367230",
        "why": "trade show literature",
    },
]

# priority = layer * 10 + number of conditions
for comp in RULES_V2:
    for r in RULES_V2[comp]:
        r["gl"] = normalize_gl(r["gl"])
        r["priority"] = r["layer"] * 10 + len(r["when"])
        for cols, op, val in r["when"]:
            assert op in OPS, r["id"]
print({k: len(v) for k, v in RULES_V2.items()})

In [ ]:
# adjustment-type lines (shipping charge corrections etc.) follow the original shipment
ADJUSTMENT_TYPES = ["SCC", "CHBK", "RADJ", "ADC", "GCC"]
PROTECTED_RULES = ["C-SVC", "C-TAX"]  # these keep their own GL


def inherit_from_tracking(raw, gl, rule, ledger=None):
    gl = gl.copy()
    rule = rule.copy()
    trk = get_col(raw, "N").values
    adj = np.isin(get_col(raw, "AJ").values, ADJUSTMENT_TYPES)

    base = pd.DataFrame({"N": trk, "GL": gl, "R": rule})[~adj & (trk != "")]
    src_rule = base.drop_duplicates("N").set_index("N")["R"]
    if ledger is not None and len(ledger) > 0:
        base = pd.concat([base, ledger[["N", "GL"]]])
    n_gl = base.groupby("N")["GL"].nunique()
    one = (
        base[base["N"].isin(n_gl[n_gl == 1].index)]
        .drop_duplicates("N")
        .set_index("N")["GL"]
    )

    new_gl = pd.Series(trk).map(one).values
    ok = adj & pd.notna(new_gl) & ~np.isin(rule, PROTECTED_RULES)
    gl[ok] = new_gl[ok]
    rule[ok] = "TRACKING"
    from_rule = np.where(ok, pd.Series(trk).map(src_rule).fillna("history").values, "")
    multi = adj & pd.Series(trk).isin(n_gl[n_gl > 1].index).values
    return gl, rule, from_rule, multi

## When to write "review"

A row gets `review` if any of these is true:
- the UPS account is new
- rules with the same priority give different GLs
- only the default rule matched, but the sender is not the company
- the rule was less than 90% accurate on the last answer key
- it's an adjustment line, and the original shipment was coded by a person / user id (the corrections often went to a different account)
- the tracking # had more than one GL before

The lists below are based on the 10/02/26 answer key.

In [ ]:
DEFAULT_RULES = ["C-DEF", "B-DEF"]
LOW_CONFIDENCE_RULES = ["C-OUT-TRB", "C-IN-RET", "C-RMA"]
WEAK_TRACKING_SOURCES = [
    "C-IT-AM",
    "C-RMA-AM",
    "C-L528-AM",
    "C-SVC-AM",
    "C-SHOW",
    "B-IN",
    "B-SAMPLE",
]

KNOWN_ACCOUNTS = {
    "CSI": [
        "000014467X",
        "00001E2419",
        "00001Y481Y",
        "000020807Y",
        "000020809A",
        "000020809V",
        "00002E37F4",
        "000035A3V9",
        "00003Y9372",
        "00007VW354",
        "0000AW1938",
        "0000AW3986",
        "0000H4B189",
        "0000H93A28",
        "0000R398E5",
        "0000V396R3",
        "0000XH8253",
        "25518",
        "38320",
    ],
    "Belimo": [
        "000088W491",
        "00001R1590",
        "00001X8Y50",
        "00009V1790",
        "543548689",
        "97644",
    ],
}


def assign_gl(raw, company, ledger=None):
    res = match_rules(raw, RULES_V2[company])
    gl, rule, from_rule, multi = inherit_from_tracking(
        raw, res["gl"].values.astype(object), res["rule"].values.astype(object), ledger
    )

    reasons = [[] for _ in range(len(raw))]

    def flag(mask, text):
        for i in np.where(mask)[0]:
            reasons[i].append(text)

    sender_own = (
        get_col(raw, "BP")
        .str.contains(make_pattern(OWN[company]), case=False, regex=True)
        .values
    )
    flag(~np.isin(get_col(raw, "C").values, KNOWN_ACCOUNTS[company]), "new UPS account")
    flag(
        res["conflict"].values & (rule != "TRACKING"),
        "rules at the same priority disagree",
    )
    flag(
        np.isin(rule, DEFAULT_RULES) & ~sender_own,
        "only the default rule matched and sender is not the company",
    )
    flag(
        np.isin(rule, LOW_CONFIDENCE_RULES),
        "rule was not very accurate on the last answer key",
    )
    flag(
        np.isin(from_rule, WEAK_TRACKING_SOURCES),
        "original shipment was coded by person/user, corrections often differ",
    )
    flag(multi & (rule != "TRACKING"), "tracking # had more than one GL before")

    need = np.array([len(r) > 0 for r in reasons])
    why = {r["id"]: r.get("why", "") for r in RULES_V2[company]}
    why["TRACKING"] = "same GL as the original shipment (same tracking #)"
    return pd.DataFrame(
        {
            "GL_Code": np.where(need, REVIEW_LABEL, gl),
            "Suggested_GL": gl,
            "Status": np.where(need, "review", "auto"),
            "Review_Reason": ["; ".join(r) for r in reasons],
            "Rule": rule,
            "Inherited_From": from_rule,
            "Why": pd.Series(rule).map(why).values,
        }
    )

## Run on input/

In [ ]:
SHOW_COLS = {
    "C": "Account",
    "L": "Ship Date",
    "N": "Tracking",
    "P": "Ref 1",
    "Q": "Ref 2",
    "AJ": "Txn Type",
    "AM": "User ID",
    "AQ": "Charge Cat",
    "AT": "Charge Desc",
    "BB": "Net Amt",
    "BP": "Sender",
    "BS": "Sender City",
    "BW": "Recv Attn",
    "BX": "Receiver",
    "CA": "Recv City",
    "CD": "Recv Country",
}


def ledger_path(company):
    return HISTORY_DIR / f"tracking_ledger_{company}.csv"


def load_ledger(company):
    p = ledger_path(company)
    if p.exists():
        return pd.read_csv(p, dtype=str, keep_default_na=False)
    return None


def process_file(path):
    path = Path(path)
    company = detect_company(path)
    raw = read_ups_csv(path)
    res = assign_gl(raw, company, load_ledger(company))

    m = col_idx("M")
    if KEEP_EXISTING_M:
        existing = raw[m].map(normalize_gl).values
        has = existing != ""
        res.loc[has, "GL_Code"] = existing[has]
        res.loc[has, "Status"] = "existing"
        res.loc[has, "Review_Reason"] = ""

    out = raw.copy()
    out[m] = res["GL_Code"].values
    stem = path.stem.replace("_Not_Completed", "")
    out_csv = OUTPUT_DIR / f"{stem}_GL.csv"
    out.to_csv(out_csv, header=False, index=False, encoding=raw.attrs["encoding"])

    table = pd.DataFrame({"Row": np.arange(1, len(raw) + 1)})
    for col, name in SHOW_COLS.items():
        if col_idx(col) < raw.shape[1]:
            table[f"{col} {name}"] = raw[col_idx(col)].values
    table = pd.concat([table, res], axis=1)
    to_review = table[table["Status"] == "review"].copy()
    to_review.insert(1, "Final_GL", "")  # fill this in by hand

    out_xlsx = OUTPUT_DIR / f"{stem}_GL_review.xlsx"
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as xw:
        to_review.to_excel(xw, sheet_name="To_Review", index=False)
        table.to_excel(xw, sheet_name="All_Rows", index=False)
        table.groupby(["Status", "GL_Code"]).size().rename(
            "Rows"
        ).reset_index().to_excel(xw, sheet_name="Summary", index=False)
        for ws in xw.book.worksheets:
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions

    return {
        "company": company,
        "path": path,
        "raw": raw,
        "result": res,
        "csv": out_csv,
        "xlsx": out_xlsx,
    }


runs = []
files = sorted(p for p in INPUT_DIR.glob("*.csv") if not is_answer_key(p))
if not files:
    print("no csv files in input/")
for f in files:
    r = process_file(f)
    runs.append(r)
    counts = r["result"]["Status"].value_counts()
    print(f"{f.name} [{r['company']}] {len(r['raw'])} rows")
    print(
        f"  auto: {counts.get('auto', 0)}, review: {counts.get('review', 0)}, existing: {counts.get('existing', 0)}"
    )
    print(f"  saved {r['csv'].name} and {r['xlsx'].name}")

In [ ]:
# why rows went to review
for r in runs:
    res = r["result"]
    print(r["path"].name)
    print(
        res.loc[res["Status"] == "review", "Review_Reason"]
        .str.split("; ")
        .explode()
        .value_counts()
        .to_string(),
        "\n",
    )

## Check against answer key (optional)
Only runs if the matching `_Completed_` file is in `input/`.

In [ ]:
for r in runs:
    key = INPUT_DIR / r["path"].name.replace("Not_Completed", "Completed")
    if key == r["path"] or not key.exists():
        print(r["path"].name, ": no answer key")
        continue
    truth = read_ups_csv(key)[col_idx("M")].map(normalize_gl).values
    res = r["result"]
    auto = res["Status"].values != "review"
    print(r["path"].name)
    print(
        "  auto rows: {}, accuracy {:.2%}".format(
            auto.sum(), (res["GL_Code"].values[auto] == truth[auto]).mean()
        )
    )
    print(
        "  review rows: {} ({:.1%}), suggested GL was right for {:.1%}".format(
            (~auto).sum(),
            (~auto).mean(),
            (res["Suggested_GL"].values[~auto] == truth[~auto]).mean(),
        )
    )
    wrong = auto & (res["GL_Code"].values != truth)
    if wrong.any():
        display(
            pd.DataFrame(
                {
                    "rule": res["Rule"][wrong],
                    "filled": res["GL_Code"][wrong],
                    "answer": truth[wrong],
                }
            )
            .value_counts()
            .rename("rows")
            .reset_index()
        )

## Write manual answers back (optional)
Fill `Final_GL` in the To_Review sheet, save the file, then set `APPLY_MANUAL = True` and run this cell.

In [ ]:
APPLY_MANUAL = False


def apply_manual_review(r):
    rv = pd.read_excel(r["xlsx"], sheet_name="To_Review", dtype=str).fillna("")
    filled = rv[rv["Final_GL"].str.strip() != ""]
    rows = filled["Row"].astype(int).values - 1
    out = read_ups_csv(r["csv"])
    out.loc[rows, col_idx("M")] = filled["Final_GL"].map(normalize_gl).values
    out.to_csv(r["csv"], header=False, index=False, encoding=out.attrs["encoding"])
    r["result"].loc[rows, "GL_Code"] = filled["Final_GL"].map(normalize_gl).values
    r["result"].loc[rows, "Status"] = "manual"
    print(
        f"{r['csv'].name}: wrote {len(filled)} rows, {(out[col_idx('M')] == REVIEW_LABEL).sum()} still review"
    )


if APPLY_MANUAL:
    for r in runs:
        apply_manual_review(r)

## Save tracking history (optional)
Shipping corrections often show up in a later bill. Saving tracking # -> GL lets those inherit the GL. Only run this after the results are checked.

In [ ]:
UPDATE_LEDGER = False


def update_ledger(r):
    res = r["result"]
    trk = get_col(r["raw"], "N").values
    keep = (
        res["Status"].isin(["auto", "manual", "existing"]).values
        & (res["Rule"].values != "TRACKING")
        & (trk != "")
    )
    new = pd.DataFrame(
        {
            "N": trk[keep],
            "GL": res["GL_Code"].values[keep],
            "source_file": r["path"].name,
        }
    )
    old = load_ledger(r["company"])
    led = pd.concat([old, new]) if old is not None else new
    led = led.drop_duplicates(["N", "GL"])
    led.to_csv(ledger_path(r["company"]), index=False)
    print(r["company"], "ledger now has", led["N"].nunique(), "tracking numbers")


if UPDATE_LEDGER:
    for r in runs:
        update_ledger(r)